<a href="https://colab.research.google.com/github/shyampandey263/langchain-structured-output/blob/main/review_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product Review Analyzer

Extracts structured data (themes, summary, sentiment, pros, cons and reviewer name) from a free-text product review using LangChain's structured output with OpenAI.

In [1]:
!pip install -q langchain langchain-openai python-dotenv

## 1. Setup
Installs the packages and asks for your OpenAI API (Application Programming Interface) key. The key is never saved in the notebook.

In [2]:
   import os, getpass
   os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste OpenAI API key: ").strip()

Paste OpenAI API key: ··········


In [3]:
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated, Optional, Literal
model = ChatOpenAI(model="gpt-4o-mini")

## 2. Define the output schema
A `TypedDict` describes the fields the model must return. The descriptions guide the model, and `Literal` limits sentiment to fixed values.

In [4]:
class Review(TypedDict):
    key_themes: Annotated[list[str], "All key themes discussed in the review"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg", "neutral"], "Overall sentiment of the review"]
    pros: Annotated[Optional[list[str]], "All the pros as a list"]
    cons: Annotated[Optional[list[str]], "All the cons as a list"]
    name: Annotated[Optional[str], "Name of the reviewer"]

structured_model = model.with_structured_output(Review)

## 3. Run it on a review
Paste any review into `review_text`. The result is a Python dictionary with the fields defined above.

In [5]:
review_text = """I bought the Redmi Note 13 last month.
The AMOLED display is bright and smooth, and the battery easily lasts a full day.
The camera is good in daylight but weak at night.
It heats up while gaming, and the pre-installed apps are annoying. For the price, it is great value.
Review by Shyam
"""

result = structured_model.invoke(review_text)
result

{'name': 'Shyam',
 'summary': 'The Redmi Note 13 offers great value with its bright AMOLED display and long-lasting battery, but has camera limitations in low light and heating issues during gaming.',
 'sentiment': 'pos',
 'pros': ['Bright and smooth AMOLED display',
  'Battery lasts a full day',
  'Great value for the price'],
 'cons': ['Camera performance weak at night',
  'Heats up during gaming',
  'Annoying pre-installed apps'],
 'key_themes': ['Display quality',
  'Battery life',
  'Camera performance',
  'Gaming performance',
  'Value for money']}

## 4. Same schema with Pydantic
Pydantic validates the output at runtime. The result is an object (`result.name`) instead of a plain dictionary.

In [6]:
from pydantic import BaseModel, Field

class ReviewPydantic(BaseModel):
    key_themes: list[str] = Field(description="All key themes discussed in the review")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg", "neutral"] = Field(description="Overall sentiment of the review")
    pros: Optional[list[str]] = Field(default=None, description="All the pros as a list")
    cons: Optional[list[str]] = Field(default=None, description="All the cons as a list")
    name: Optional[str] = Field(default=None, description="Name of the reviewer")

structured_model_pydantic = model.with_structured_output(ReviewPydantic)

In [7]:
result_pydantic = structured_model_pydantic.invoke(review_text)

print(result_pydantic.name)          # attribute access
print(result_pydantic.sentiment)
print(result_pydantic.model_dump())  # convert to a normal dictionary

Shyam
pos
{'key_themes': ['AMOLED display quality', 'Battery life', 'Camera performance', 'Gaming performance', 'Pre-installed apps', 'Value for money'], 'summary': "The Redmi Note 13 has a bright AMOLED display and good battery life, but struggles with night photography and heats up during gaming. It's seen as great value for its price despite some drawbacks.", 'sentiment': 'pos', 'pros': ['Bright and smooth AMOLED display', 'Good battery life (lasts a full day)', 'Great value for the price'], 'cons': ['Camera performance is weak at night', 'Heats up while gaming', 'Annoying pre-installed apps'], 'name': 'Shyam'}
